<a href="https://colab.research.google.com/github/yaesur/business_python/blob/%EC%B2%AD%EB%85%84%EB%A7%A4%EC%9E%85%EC%9E%84%EB%8C%80%EC%A3%BC%ED%83%9D/%EC%A7%80%EC%97%AD%EC%84%A0%ED%98%B8%EB%8F%84_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import numpy as np

file = pd.ExcelFile('커트라인 데이터.xlsx')
sheets = file.sheet_names

# 시트별 컬럼 설정
level_cols = [5, 5, 5, 5, 5, 5, 5, 4, 4, 7]
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
result = []

# 데이터 로드
for i, name in enumerate(sheets):
    df = pd.read_excel(file, sheet_name=name, skiprows=2, header=None)
    data = df.iloc[:, [1, level_cols[i], score_cols[i]]]
    data.columns = ['자치구', '순위', '점수']
    result.append(data)

final = pd.concat(result).dropna()

final['순위'] = final['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
final['순위'] = pd.to_numeric(final['순위'], errors='coerce')
final['점수'] = final['점수'].astype(str).str.replace('점', '', regex=False).str.strip()
final['점수'] = pd.to_numeric(final['점수'], errors='coerce')
final['자치구'] = final['자치구'].str.strip()

In [29]:
# 1. 자치구별 전체 데이터 개수(공급 주택 수 트렌드) 계산
total_counts = final.groupby('자치구').size()

# 2. 자치구별 '1순위'에서 마감된 데이터 개수 계산
first_priority_counts = final[final['순위'] == 1].groupby('자치구').size()

# 3. 자치구별 1순위 비율 계산 (1순위 데이터가 없는 구는 0으로 채움)
priority_ratio = (first_priority_counts / total_counts).fillna(0)

# 4. 분석 및 확인을 위한 데이터프레임 생성
district_analysis = priority_ratio.reset_index(name='1순위_비율')
# 보기 편하게 전체 대비 2순위 이하 비율도 함께 계산
district_analysis['2순위_이상_비율'] = 1 - district_analysis['1순위_비율']


# 5. [핵심] 제안하신 기준에 따른 ABC 등급 분류 함수 정의
# 실제 데이터 분포를 보면서 아래 커트라인 비율(0.8, 0.3)은 자유롭게 조정 가능합니다.
def determine_abc_grade(ratio):
    if ratio >= 0.85:      # 1순위 마감 비율이 85% 이상 -> 1순위 대부분 차지
        return 'A'
    elif ratio >= 0.40:    # 1순위 마감 비율이 40% ~ 85% 사이 -> 1, 2순위 혼재
        return 'B'
    else:                  # 1순위 마감 비율이 30% 미만 -> 2순위 대부분 차지
        return 'C'

# 등급 부여
district_analysis['새로운_등급'] = district_analysis['1순위_비율'].apply(determine_abc_grade)


# 6. 결과 출력 (어떤 구가 어떻게 분류되었는지 먼저 확인)
print("======= 자치구별 순위 분포 기반 등급 분류 결과 =======")
print(district_analysis.sort_values(by='1순위_비율', ascending=False).to_string(index=False))
print("-" * 60)

raw_grade_map = district_analysis.groupby('새로운_등급')['자치구'].apply(list).to_dict()

# 데이터가 없는 등급이 있더라도 에러가 나지 않도록 처리하며 A, B, C 순으로 정렬
grade_map = {grade: raw_grade_map[grade] for grade in ['A', 'B', 'C'] if grade in raw_grade_map}

import pprint
pprint.pprint(grade_map)

def get_grade(district):
  for grade, districts in grade_map.items():
    if district in districts:
      return grade

final['등급'] = final['자치구'].apply(get_grade)
grade_avg = final.groupby('등급')['점수'].mean().round(4).reset_index()

grade_avg

======= 자치구별 순위 분포 기반 등급 분류 결과 =======
 자치구   1순위_비율  2순위_이상_비율 새로운_등급
 강남구 1.000000   0.000000      A
 종로구 1.000000   0.000000      A
 마포구 1.000000   0.000000      A
 노원구 1.000000   0.000000      A
  중구 1.000000   0.000000      A
 성북구 0.866667   0.133333      A
 성동구 0.857143   0.142857      A
 서초구 0.846154   0.153846      B
 광진구 0.775510   0.224490      B
동대문구 0.774194   0.225806      B
 동작구 0.769231   0.230769      B
 관악구 0.742424   0.257576      B
 송파구 0.713043   0.286957      B
 도봉구 0.692308   0.307692      B
 강서구 0.600000   0.400000      B
서대문구 0.594595   0.405405      B
영등포구 0.578313   0.421687      B
 양천구 0.500000   0.500000      B
 중랑구 0.477273   0.522727      B
 강북구 0.462687   0.537313      B
 강동구 0.437500   0.562500      B
 은평구 0.378378   0.621622      C
 구로구 0.327869   0.672131      C
 금천구 0.320513   0.679487      C
------------------------------------------------------------
{'A': ['강남구', '노원구', '마포구', '성동구', '성북구', '종로구', '중구'],
 'B': ['강동구',
       '강북구',
       '강서구',
  

,등급,점수
0,A,6.3525
1,B,6.0761
2,C,5.5726


In [32]:
data = {'등급_숫자': [2,1,0], '점수': [6.3525, 6.0761,5.5726]} #A=2, B=1, C=0
df_corr = pd.DataFrame(data)

correlation = df_corr['등급_숫자'].corr(df_corr['점수'])

print(f"상관계수: {correlation:.4f}")
print("1에 가까울수록 등급과 점수가 정확히 비례한다는 뜻입니다.")

상관계수: 0.9862
1에 가까울수록 등급과 점수가 정확히 비례한다는 뜻입니다.
